Installing Phase:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
base_path_sess2 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/2nd_session/extractedvein'

NUM_SUBJECTS = 123
NUM_FINGERS = 4
IMAGES_PER_SESSION = 4
IMAGE_SIZE = (100, 300)
FUSION_METHOD = 'vstack'

# ==== STORAGE ====
fused_images = []
fused_labels = []

# ==== STEP 1: LOAD AND FUSE IMAGES ====
def process_session_strategy1(base_path, session_label):
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc=f"Session {session_label}"):
        for img_idx in range(1, IMAGES_PER_SESSION + 1):
            images_2d = []

            print(f"\n➡️ Subject {subject_id:03d} - Image {img_idx} - Session {session_label}")

            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                folder_path = os.path.join(base_path, folder_name)
                img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")

                print(f"  📥 Loading finger {finger_id} from: {img_path}")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"  ❌ Missing image: {img_path}")
                    continue

                print(f"  ✅ Original image shape: {img.shape}")
                img = cv2.resize(img, IMAGE_SIZE)
                print(f"  📐 Resized to: {img.shape}")

                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                images_2d.append(img_norm)

            if len(images_2d) == 4:
                print(f"🔄 Fusing 4 finger images using method: {FUSION_METHOD}")

                if FUSION_METHOD == 'vstack':
                    fused_img = np.vstack(images_2d)
                elif FUSION_METHOD == 'hstack':
                    fused_img = np.hstack(images_2d)
                elif FUSION_METHOD == '2x2':
                    top = np.hstack([images_2d[0], images_2d[1]])
                    bottom = np.hstack([images_2d[2], images_2d[3]])
                    fused_img = np.vstack([top, bottom])
                else:
                    raise ValueError("Unsupported fusion method.")

                print(f"  📸 Fused image shape: {fused_img.shape}")
                fused_images.append(fused_img)
                fused_labels.append(f"{subject_id:03d}_img{img_idx}_s{session_label}")
            else:
                print(f"  ⚠️ Skipped fusion (found {len(images_2d)} valid images instead of 4)")

# ==== APPLY STRATEGY 1 ====
process_session_strategy1(base_path_sess1, session_label=1)
process_session_strategy1(base_path_sess2, session_label=2)

print(f"\n✅ Total fused samples: {len(fused_images)}")
print(f"✅ Example fused image shape: {fused_images[0].shape}")
fused_labels = np.array(fused_labels)

# ==== STEP 2: COMPUTE 2DPCA ====
def compute_2dpca_projection(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:  # print details for first 3 only
            print(f"  ➕ Image {i+1} contribution added to covariance")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ Computed projection matrix shape: {eig_vecs.shape}")
    return eig_vecs

# ==== STEP 3: PROJECT IMAGES ====
num_components = 137
W = compute_2dpca_projection(fused_images, num_components)

projected_features = []
for i, img in enumerate(fused_images):
    feat = img @ W
    projected_features.append(feat)
    if i < 3:
        print(f"🧮 Sample {i+1} projected shape: {feat.shape}")

# ==== STEP 4: OPTIONAL — FLATTEN FOR CLASSIFIER ====
flat_features = np.array([feat.flatten() for feat in projected_features])
print(f"\n✅ Flattened feature matrix shape: {flat_features.shape}")


Training:

In [ ]:
test_data = []
test_labels = []
test_paths = []

for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Preparing test data"):
    for img_idx in [5, 6]:  # P1 protocol test images
        for session_label, base_path in [(1, base_path_sess1), (2, base_path_sess2)]:
            images_2d = []
            current_paths = []

            for finger_id in range(1, NUM_FINGERS + 1):
                folder_name = f"vein{subject_id:03d}_{finger_id}"
                img_path = os.path.join(base_path, folder_name, f"{img_idx:02d}.jpg")

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"⚠️ Missing: {img_path}")
                    continue

                print(f"✅ Using: {img_path}")
                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                images_2d.append(img_norm)
                current_paths.append(img_path)

            if len(images_2d) == 4:
                fused_img = np.vstack(images_2d)
                test_data.append(fused_img)
                test_labels.append(f"{subject_id:03d}_img{img_idx}_s{session_label}")
                test_paths.append(current_paths)
            else:
                print(f"⚠️ Incomplete: Subject {subject_id}, Image {img_idx}, Session {session_label}")

test_data = np.array(test_data)
test_labels = np.array(test_labels)
# ==== STEP X: PROJECT TEST IMAGES USING 2DPCA ====
proj_test_features = [img @ W for img in test_data]  # img shape: (400, 300)
flat_test_features = np.array([f.flatten() for f in proj_test_features])

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")

Banchmarking:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_img5_s2"

    # Compute Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # Get the nearest neighbor
    min_index = np.argmin(distances)
    predicted_label = fused_labels[min_index]  # e.g., "005_img2_s2"

    print(f"\nTest sample {i+1}:")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")

    # Compare subject ID and session
    pred_id, pred_session = predicted_label.split('_')[0], predicted_label.split('_')[-1]
    true_id, true_session = true_label.split('_')[0], true_label.split('_')[-1]

    if pred_id == true_id and pred_session == true_session:
        correct_matches += 1
        print("  🟢 Match (ID & Session correct)")
    else:
        print("  🔴 Mismatch")

# Compute accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final recognition accuracy: {accuracy:.2f}%")


Benchmarking 2:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance (Match: Subject ID only)...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_img5_s2"

    # Compute Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # Get the nearest neighbor
    min_index = np.argmin(distances)
    predicted_label = fused_labels[min_index]  # e.g., "005_img2_s2"

    print(f"\nTest sample {i+1}:")
    print(f"  🎯 Predicted → {predicted_label}")
    print(f"  ✅ Actual    → {true_label}")

    # Extract only the subject ID (first part before "_img")
    pred_id = predicted_label.split('_')[0]
    true_id = true_label.split('_')[0]

    if pred_id == true_id:
        correct_matches += 1
        print("  ✅ Match (ID correct, session ignored)")
    else:
        print("  ❌ Mismatch")

# Compute accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final recognition accuracy (Person only): {accuracy:.2f}%")
